In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
# load all environment variables
load_dotenv()
# Load groq API key into environment variable
os.environ["GROK_API_KEY"] = os.getenv("GROK_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
# LangSmith Tracking configuration
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY_LangTrans"] = os.getenv("LANGCHAIN_API_KEY_LangTrans")
os.environ["LANGCHAIN_TRACKING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
groq_api_key = os.getenv("GROK_API_KEY")
huggingfacehub_api_token = os.getenv("HF_TOKEN")
llm_model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
llm_model

e:\Development\00-Repos\MyRepo\GenAiApps\LangChainvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018668DA8D50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018669ECF990>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [6]:
from langchain_core.documents import Document

documents = [
    Document(page_content="Dogs are great companions. known for their loyalty and affection.", metadata={"source": "mammal-pets-doc"}),
    Document(page_content="Cats are independent and curious animals that often enjoy their own space.", metadata={"source": "groq.txt"}),
    Document(page_content="Goldfish are small freshwater fish that are popular as pets.", metadata={"source": "groq.txt"}),
    Document(page_content="Parrots are colorful birds known for their ability to mimic sounds.", metadata={"source": "groq.txt"}),
]

In [9]:
from langchain_chroma import Chroma
vector_store = Chroma.from_documents(
    documents,
    embedding=embedding_model,
)


In [10]:
vector_store.similarity_search("Cat")

[Document(id='66a389f4-e746-4c4c-8f20-fec68cd7a788', metadata={'source': 'groq.txt'}, page_content='Cats are independent and curious animals that often enjoy their own space.'),
 Document(id='62ef3557-45f7-46c6-9683-17c6fbeea923', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions. known for their loyalty and affection.'),
 Document(id='9f2addd1-d25a-4bea-9d31-781ad118a9dc', metadata={'source': 'groq.txt'}, page_content='Parrots are colorful birds known for their ability to mimic sounds.'),
 Document(id='50f16851-bd3d-4a11-8fe7-8fe6d11a8d5c', metadata={'source': 'groq.txt'}, page_content='Goldfish are small freshwater fish that are popular as pets.')]

In [11]:
## asynch queury example
await vector_store.asimilarity_search("Cat")

[Document(id='66a389f4-e746-4c4c-8f20-fec68cd7a788', metadata={'source': 'groq.txt'}, page_content='Cats are independent and curious animals that often enjoy their own space.'),
 Document(id='62ef3557-45f7-46c6-9683-17c6fbeea923', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions. known for their loyalty and affection.'),
 Document(id='9f2addd1-d25a-4bea-9d31-781ad118a9dc', metadata={'source': 'groq.txt'}, page_content='Parrots are colorful birds known for their ability to mimic sounds.'),
 Document(id='50f16851-bd3d-4a11-8fe7-8fe6d11a8d5c', metadata={'source': 'groq.txt'}, page_content='Goldfish are small freshwater fish that are popular as pets.')]

In [12]:
## similrity search with score
vector_store.similarity_search_with_score("Cat")

[(Document(id='66a389f4-e746-4c4c-8f20-fec68cd7a788', metadata={'source': 'groq.txt'}, page_content='Cats are independent and curious animals that often enjoy their own space.'),
  0.9610071778297424),
 (Document(id='62ef3557-45f7-46c6-9683-17c6fbeea923', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions. known for their loyalty and affection.'),
  1.5100563764572144),
 (Document(id='9f2addd1-d25a-4bea-9d31-781ad118a9dc', metadata={'source': 'groq.txt'}, page_content='Parrots are colorful birds known for their ability to mimic sounds.'),
  1.6902141571044922),
 (Document(id='50f16851-bd3d-4a11-8fe7-8fe6d11a8d5c', metadata={'source': 'groq.txt'}, page_content='Goldfish are small freshwater fish that are popular as pets.'),
  1.7310606241226196)]

In [14]:
## Retrievers
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vector_store.similarity_search).bind(k=1)
retriever.batch(["Cat", "Dog"])  


[[Document(id='66a389f4-e746-4c4c-8f20-fec68cd7a788', metadata={'source': 'groq.txt'}, page_content='Cats are independent and curious animals that often enjoy their own space.')],
 [Document(id='62ef3557-45f7-46c6-9683-17c6fbeea923', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions. known for their loyalty and affection.')]]

In [15]:
## Retrievers
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 1})
retriever.batch(["Cat", "Dog"])  

[[Document(id='66a389f4-e746-4c4c-8f20-fec68cd7a788', metadata={'source': 'groq.txt'}, page_content='Cats are independent and curious animals that often enjoy their own space.')],
 [Document(id='62ef3557-45f7-46c6-9683-17c6fbeea923', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions. known for their loyalty and affection.')]]

In [22]:
## RAG Chain Example
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer the question based on the context below.
question: {question}
Context: {context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm_model
response = chain.invoke("Tell me about cats?")
print(response.content)

Based on the provided context, here's what I can tell you about cats:

Cats are independent and curious animals that often enjoy their own space.
